In [2]:
import pandas as pd

df = pd.read_csv("/content/CO2 Emissions_Canada.csv")
df.head()

,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
0,ACURA,ILX,COMPACT,2.0,4,AS5,Z,9.9,6.7,8.5,33,196
1,ACURA,ILX,COMPACT,2.4,4,M6,Z,11.2,7.7,9.6,29,221
2,ACURA,ILX HYBRID,COMPACT,1.5,4,AV7,Z,6.0,5.8,5.9,48,136
3,ACURA,MDX 4WD,SUV - SMALL,3.5,6,AS6,Z,12.7,9.1,11.1,25,255
4,ACURA,RDX AWD,SUV - SMALL,3.5,6,AS6,Z,12.1,8.7,10.6,27,244


In [3]:
X = df[["Engine Size(L)", "Cylinders", "Fuel Consumption Comb (L/100 km)"]]
y = df["CO2 Emissions(g/km)"]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

In [7]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)

y_pred_lasso = lasso.predict(X_test_scaled)

In [8]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

param_grid = {"alpha": [0.01, 0.1, 1, 10, 100]}

ridge = Ridge()

grid = GridSearchCV(ridge, param_grid, cv=5)
grid.fit(X_train_scaled, y_train)

best_ridge = grid.best_estimator_
best_alpha = grid.best_params_["alpha"]

y_pred_ridge = best_ridge.predict(X_test_scaled)

print("Best Alpha:", best_alpha)

Best Alpha: 10


In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate(name, y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"----- {name} -----")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2  :", r2)
    print()

    return mae, rmse, r2

In [10]:
results = {}

results["Linear Regression"] = evaluate("Linear Regression", y_test, y_pred_lr)
results["Lasso Regression"] = evaluate("Lasso Regression", y_test, y_pred_lasso)
results["Best Ridge Regression"] = evaluate("Best Ridge Regression", y_test, y_pred_ridge)

results_df = pd.DataFrame(results, index=["MAE", "RMSE", "R2 Score"]).T
results_df

----- Linear Regression -----
MAE : 13.517321294682649
RMSE: 20.540748085335153
R2  : 0.8773348735033225

----- Lasso Regression -----
MAE : 13.534646737575086
RMSE: 20.54241157861251
R2  : 0.8773150046187961

----- Best Ridge Regression -----
MAE : 13.543667005117833
RMSE: 20.541396384652405
R2  : 0.8773271303605337



,MAE,RMSE,R2 Score
Linear Regression,13.517321,20.540748,0.877335
Lasso Regression,13.534647,20.542412,0.877315
Best Ridge Regression,13.543667,20.541396,0.877327
